# Proyecto Sure Tomorrow: Solución Completa

Este notebook contiene la solución paso a paso a las cuatro tareas solicitadas, con comentarios detallados para Python de nivel básico-intermedio.

In [ ]:
# Imports y carga de datos
import numpy as np
import pandas as pd
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
import math

# Carga del dataset
df = pd.read_csv('insurance_us.csv')
df = df.rename(columns={
    'Gender': 'gender',
    'Age': 'age',
    'Salary': 'income',
    'Family members': 'family_members',
    'Insurance Benefit Count': 'insurance_benefits'
})

df.head()  # Vistazo inicial

## Tarea 1: Clientes Similares
Preparación de datos y función kNN.

In [ ]:
# Preparación de atributos
feature_names = ['gender', 'age', 'income', 'family_members']
df_knn = df[feature_names].copy()
df_knn['gender'] = df_knn['gender'].map({'Male': 0, 'Female': 1})
df_knn.head()

In [ ]:
# Función para obtener k vecinos
def get_knn(data, idx, k=5, metric='euclidean'):
    nbrs = NearestNeighbors(n_neighbors=k+1, metric=metric)
    nbrs.fit(data)
    distances, indices = nbrs.kneighbors(data)
    return indices[idx][1:], distances[idx][1:]

# Ejemplo
ids, dist = get_knn(df_knn, idx=0, k=5, metric='euclidean')
print('Vecinos:', ids, 'Distancias:', np.round(dist,2))

## Tarea 2: Clasificación Binaria
Predicción de probabilidad de recibir al menos una prestación.

In [ ]:
# Definición de target
df['received_benefit'] = (df['insurance_benefits'] > 0).astype(int)
print(df['received_benefit'].value_counts(normalize=True))

# División de datos
X = df_knn.values
y = df['received_benefit'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# kNN
clf_knn = KNeighborsClassifier(n_neighbors=5)
clf_knn.fit(X_train, y_train)
y_prob_knn = clf_knn.predict_proba(X_test)[:,1]
print('kNN AUC:', roc_auc_score(y_test, y_prob_knn))

# Dummy
clf_dummy = DummyClassifier(strategy='prior')
clf_dummy.fit(X_train, y_train)
y_prob_dummy = clf_dummy.predict_proba(X_test)[:,1]
print('Dummy AUC:', roc_auc_score(y_test, y_prob_dummy))

## Tarea 3: Regresión Lineal desde Cero
Implementación de regresión lineal y evaluación.

In [ ]:
# Implementación de MyLinearRegression
class MyLinearRegression:
    def fit(self, X, y):
        XtX_inv = np.linalg.inv(X.T.dot(X))
        self.coef_ = XtX_inv.dot(X.T).dot(y)
    def predict(self, X):
        return X.dot(self.coef_)

# Preparación de datos
y_reg = df['insurance_benefits'].values
X_reg = np.hstack([np.ones((len(df),1)), df_knn.values])
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

# Entrenamiento y evaluación
model = MyLinearRegression()
model.fit(Xr_train, yr_train)
y_pred = model.predict(Xr_test)
rmse = math.sqrt(mean_squared_error(yr_test, y_pred))
print(f'RMSE: {rmse:.3f}')

## Tarea 4: Ofuscación de Datos
Multiplicación por matriz invertible y recuperación.

In [ ]:
# Ofuscación reversible
X = df_knn.values
rng = np.random.default_rng(42)
# Generar matriz P invertible
while True:
    P = rng.random((X.shape[1], X.shape[1]))
    if np.linalg.det(P) != 0:
        break
X_prime = X.dot(P)
X_recov = X_prime.dot(np.linalg.inv(P))
print('Recuperación exacta:', np.allclose(X, X_recov))